# SnakeCheck: model training

Trains the 6-class model (`non_venomous`, the four venomous species, `not_a_snake`) on the **frozen, observer-based splits** made by `scripts/build_splits.py`.

- **train**: used for training only.
- **val**: picks the best epoch and the venomous threshold.
- **test**: India photos from observers the model never saw.
- **test_external**: Telangana (left out of training completely) plus your own photos.

Test and test_external are scored once, at the end, with the threshold fixed on val. Nothing is evaluated on training images.

The training code lives in `scripts/train.py` and `scripts/snake_eval.py` in the repo. This notebook imports them, so Colab and local runs train the same way.

**Before running**
1. On your computer: `python scripts/build_splits.py --materialize data/splits`, then `python scripts/zip_for_colab.py`.
2. Upload `data/snakecheck_data.zip` to Google Drive as `My Drive/snakecheck/data.zip`.
3. Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

## 1. Setup

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU found. Go to Runtime > Change runtime type > T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the training code from GitHub. Use the branch with the new pipeline until it is merged into main.
REPO = 'https://github.com/Abhinay-red/snakecheck.git'
BRANCH = 'main'

import subprocess, sys
from pathlib import Path
REPO_DIR = Path('/content/snakecheck')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
sys.path.insert(0, str(REPO_DIR / 'scripts'))

import importlib, train as sc, snake_eval as se
importlib.reload(se); importlib.reload(sc)
print('Code from', BRANCH, subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'],
                                          capture_output=True, text=True).stdout.strip())

In [ ]:
import zipfile, hashlib

DRIVE_ROOT = Path('/content/drive/MyDrive/snakecheck')
DATA_ZIP   = DRIVE_ROOT / 'data.zip'
DATA_DIR   = Path('/content/snakecheck_data')     # fast local disk
MODELS_DIR = DRIVE_ROOT / 'models'                # saved back to Drive
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_ZIP.exists(), f'data.zip not found at {DATA_ZIP}. Upload it to Google Drive first.'
if not (DATA_DIR / 'train').exists():
    print('Extracting dataset...')
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_DIR)
for s in sc.SPLITS:
    assert s in ('test_external',) or (DATA_DIR / s).is_dir(), f'{s}/ missing in data.zip; rerun build_splits.py --materialize'

frozen = DATA_DIR / 'manifests' / 'eval_frozen.csv'
EVAL_SHA = hashlib.sha256(frozen.read_bytes()).hexdigest()[:16] if frozen.exists() else 'unknown'
print('Frozen eval set:', EVAL_SHA, '(runs are only comparable when this matches)')

## 2. Configuration

In [ ]:
CLASS_NAMES, VENOMOUS = sc.load_classes(REPO_DIR / 'config' / 'dataset.yaml')
NUM_CLASSES = len(CLASS_NAMES)

P1_EPOCHS = 15
P2_EPOCHS = 40
BATCH_SIZE = sc.BATCH_SIZE

# The threshold is picked on val so that at least this share of venomous snakes is caught.
# Calling a harmless snake venomous is the safer mistake, so this is set high.
VENOMOUS_RECALL_TARGET = 0.95

print('Classes:', CLASS_NAMES)
print('Venomous:', VENOMOUS)
print('Launch gates (checked on test sets):', sc.GATES)

## 3. Data

In [ ]:
import numpy as np
tf.keras.utils.set_random_seed(sc.SEED)
splits = sc.load_splits(DATA_DIR, CLASS_NAMES, BATCH_SIZE)
class_weights = sc.class_weights(splits['train']['counts'])
print('Class weights (from train counts):')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c:<18} {class_weights[i]:.3f}')

## 4. Phase 1: head only (backbone frozen)

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime

RUN_DIR = MODELS_DIR / datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run folder:', RUN_DIR)

model, base = sc.build_model(NUM_CLASSES)
sc.compile_model(model, sc.P1_LR, NUM_CLASSES)
h1 = model.fit(splits['train']['ds'], validation_data=splits['val']['ds'], epochs=P1_EPOCHS,
               class_weight=class_weights,
               callbacks=sc.make_callbacks(RUN_DIR / 'best_phase1.keras', RUN_DIR / 'history_phase1.csv'))

def plot(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.history['loss'], label='train'); ax[0].plot(h.history['val_loss'], label='val')
    ax[0].set_title(f'{title}: loss'); ax[0].legend()
    ax[1].plot(h.history['macro_recall'], label='train'); ax[1].plot(h.history['val_macro_recall'], label='val')
    ax[1].set_title(f'{title}: macro recall'); ax[1].legend()
    plt.tight_layout(); plt.show()

plot(h1, 'Phase 1')

## 5. Phase 2: fine-tune the top backbone layers

In [ ]:
sc.unfreeze_top(base, sc.P2_UNFREEZE_TOP)
sc.compile_model(model, sc.P2_LR, NUM_CLASSES)
h2 = model.fit(splits['train']['ds'], validation_data=splits['val']['ds'], epochs=P2_EPOCHS,
               class_weight=class_weights,
               callbacks=sc.make_callbacks(RUN_DIR / 'best_phase2.keras', RUN_DIR / 'history_phase2.csv'))
plot(h2, 'Phase 2')

final_path = RUN_DIR / 'snakecheck_final.keras'
model.save(final_path)
print('Saved', final_path)

In [ ]:
# To evaluate an earlier run instead of training, set its folder here and run this cell.
# RUN_DIR = MODELS_DIR / '20261125_120000'
# sc.macro_recall_metric(NUM_CLASSES)
# model = tf.keras.models.load_model(RUN_DIR / 'snakecheck_final.keras')

## 6. Evaluation

The threshold is picked on **val**. Then val, test and test_external are scored with that same threshold.
You get per-class recall and the venomous-vs-not view (four venomous classes against harmless snakes; `not_a_snake` is reported on its own).

In [ ]:
import json
results = sc.evaluate_all(model, splits, CLASS_NAMES, VENOMOUS, VENOMOUS_RECALL_TARGET)
THRESHOLD = results['threshold']
meta = {'class_names': CLASS_NAMES, 'venomous_classes': VENOMOUS, 'threshold': THRESHOLD,
        'venomous_recall_target': VENOMOUS_RECALL_TARGET, 'eval_manifest_sha': EVAL_SHA,
        'img_size': sc.IMG_SIZE, 'input': 'RGB 0-255, resized to 224x224',
        'train_counts': dict(zip(CLASS_NAMES, map(int, splits['train']['counts'])))}
(RUN_DIR / 'config.json').write_text(json.dumps(meta, indent=2))
(RUN_DIR / 'metrics.json').write_text(json.dumps(results, indent=2, default=float))
print('Saved metrics to', RUN_DIR)

In [ ]:
import seaborn as sns
for s in ('val', 'test', 'test_external'):
    if s not in results:
        continue
    cm = results[s]['confusion_matrix']
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion matrix: {s}'); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout()
    plt.savefig(str(RUN_DIR / f'confusion_{s}.png'), dpi=120); plt.show()

In [ ]:
# Try your own photo, using the same rule the app will use.
from google.colab import files
from PIL import Image
import io, numpy as np

vmask = se.venomous_mask(CLASS_NAMES, VENOMOUS)
for fname, data in files.upload().items():
    img = Image.open(io.BytesIO(data)).convert('RGB').resize((sc.IMG_SIZE, sc.IMG_SIZE))
    x = np.expand_dims(np.asarray(img, dtype=np.float32), 0)      # 0-255, like training
    probs = model.predict(x, verbose=0)
    pred = se.predict(probs, vmask, THRESHOLD)[0]
    pv = se.p_venomous(probs, vmask)[0]
    print(f'\n{fname}: {CLASS_NAMES[pred]}  (venomous score {pv:.2f}, threshold {THRESHOLD:.2f})')
    for c, p in zip(CLASS_NAMES, probs[0]):
        print(f'  {c:<18} {p:.3f}')

## 7. TFLite export (INT8)

Runs only if the launch gates passed on **both** test sets.

In [ ]:
passed = all(results.get(s, {}).get('gates_passed', False) for s in ('test', 'test_external'))
print('Gates passed on test and test_external:', passed)
assert passed, 'Gates not passed. Do not export; look at the confusion matrices and add data in small batches.'

def representative_dataset():
    for images, _ in splits['train']['ds'].take(100):
        for img in images:
            yield [tf.expand_dims(img, 0)]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.uint8      # the app passes 0-255 pixels
conv.inference_output_type = tf.float32
tflite_int8 = conv.convert()
int8_path = RUN_DIR / 'snakecheck_int8.tflite'
int8_path.write_bytes(tflite_int8)
print(f'INT8 TFLite: {len(tflite_int8) / 1e6:.1f} MB -> {int8_path}')
assert len(tflite_int8) < 15e6, 'Model too large (PRD limit 15 MB)'

In [ ]:
# Check the TFLite model agrees with the Keras model on the external test set.
interp = tf.lite.Interpreter(model_path=str(int8_path))
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
split = splits['test_external'] if splits['test_external']['ds'] else splits['test']
agree = total = 0
for images, labels in split['ds'].take(5):
    keras_pred = model.predict(images, verbose=0).argmax(1)
    for img, kp in zip(images.numpy(), keras_pred):
        interp.set_tensor(inp['index'], np.clip(img, 0, 255).astype(np.uint8)[None])
        interp.invoke()
        agree += int(interp.get_tensor(out['index'])[0].argmax() == kp); total += 1
print(f'TFLite matches Keras on {agree}/{total} images')

import time
x = np.clip(images.numpy()[:1], 0, 255).astype(np.uint8)
times = []
for _ in range(50):
    interp.set_tensor(inp['index'], x); t0 = time.perf_counter(); interp.invoke()
    times.append((time.perf_counter() - t0) * 1000)
print(f'Latency on Colab CPU: median {np.median(times):.1f} ms, p95 {np.percentile(times, 95):.1f} ms')

## 8. Done

Everything is saved in `My Drive/snakecheck/models/<run>/`:

| File | Use |
|---|---|
| `snakecheck_final.keras` | Full Keras model |
| `snakecheck_int8.tflite` | The model for the mobile app |
| `config.json` | Classes, the venomous threshold the app must use, and which frozen eval set was used |
| `metrics.json` | Val, test and test_external results |
| `confusion_*.png` | Confusion matrices |

When you add a new data batch, train again and compare `metrics.json` on **val** with the previous run (same `eval_manifest_sha`). Keep the batch only if val gets better.